# Capstone: build a tiny diffusion model (optional)
**LLMs, Animated · Capstone 2**

You'll build a complete, class-conditional image diffusion model from scratch in PyTorch, on 16×16 pictures of four shapes that the notebook draws itself (no downloads):

1. the linear noise schedule and one-jump noising $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$,
2. a sinusoidal **time embedding** for the denoiser,
3. the training loss, with **label dropout** so one network learns both conditional and unconditional predictions,
4. the 1,000-step **DDPM** sampler with **classifier-free guidance**,
5. a deterministic **DDIM** sampler that needs only 20 steps.

Each exercise has a `# TODO` cell followed by a **check** cell with `assert`s. Solutions are at the very bottom.

**Runtime.** With the defaults everything runs on a CPU in about two minutes; most of it is training. On Colab, *Runtime → Change runtime type → T4 GPU* and `LONG = True` trains a bigger model for longer (a few minutes on the GPU) for visibly sharper samples. The checks pass either way.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

LONG = False                       # True: bigger model, 20,000 steps (use a GPU runtime)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
if device == "cpu":
    torch.set_num_threads(2)       # small models are fastest on a couple of threads
print("device:", device, "| LONG:", LONG)

S, N_CLASSES, NULL = 16, 4, 4      # image size, classes, and the extra "no label" index for CFG
CLASSES = ["disk", "square", "triangle", "plus"]

def make_shapes(n, seed=0):
    # white shapes on black, pixel values exactly -1 or +1, centred with up to 2 px of jitter
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:S, 0:S] + 0.5
    X = np.full((n, S, S), -1.0, np.float32); y = rng.integers(0, N_CLASSES, n)
    for i in range(n):
        r = rng.uniform(3.5, 5.5); cx, cy = S / 2 + rng.uniform(-2, 2, 2)
        if y[i] == 0:   m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r * r
        elif y[i] == 1: m = (abs(xx - cx) <= 0.8 * r) & (abs(yy - cy) <= 0.8 * r)
        elif y[i] == 2: m = (yy <= cy + 0.8 * r) & (yy >= cy - r) & (abs(xx - cx) <= 0.62 * (yy - cy + r))
        else:
            w = max(1.0, 0.3 * r)
            m = ((abs(xx - cx) <= w) & (abs(yy - cy) <= r)) | ((abs(yy - cy) <= w) & (abs(xx - cx) <= r))
        X[i][m] = 1.0
    return torch.tensor(X).unsqueeze(1), torch.tensor(y)

X, y = make_shapes(5000, seed=0)          # training set
X_val, y_val = make_shapes(1000, seed=1)  # for validation loss and the judge

def show(imgs, title=None, n=16):
    fig, ax = plt.subplots(1, n, figsize=(n * 0.8, 1.0))
    for i in range(n):
        ax[i].imshow(imgs[i, 0].detach().cpu().clamp(-1, 1), cmap="gray", vmin=-1, vmax=1); ax[i].axis("off")
    if title: fig.suptitle(title, y=1.15, fontsize=10)
    plt.show()

show(X, "training data")
print(X.shape, "pixel values:", X.unique().tolist())

## 1 · The noise schedule and one-jump noising
Use $T = 1000$ steps with $\beta_t$ rising **linearly** from $10^{-4}$ to $0.02$ (as in G1.3), $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s \le t} \alpha_s$. Index the tensors `0 … T-1` (index `t` holds the paper's step `t + 1`).

Then write `q_sample(x0, t, eps)`, which returns $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ for a **batch**: `t` is a tensor of shape `(B,)`, so reshape the coefficients to `(B, 1, 1, 1)` before multiplying.

In [ ]:
T = 1000
# TODO: betas, alphas, alpha_bar: float tensors of shape (T,)
betas = None
alphas = None
alpha_bar = None

def q_sample(x0, t, eps):
    # TODO: one-jump noising for a batch; t has shape (B,)
    raise NotImplementedError("Exercise 1")

**Check 1.** The lesson's numbers: $\bar\alpha_{250} \approx 0.524$, $\bar\alpha_{500} \approx 0.079$, $\bar\alpha_{1000} \approx 4 \times 10^{-5}$. The mean of $x_t$ shrinks by $\sqrt{\bar\alpha_t}$, and by the last step $x_t$ is plain $\mathcal{N}(0, 1)$ noise.

In [ ]:
assert betas.shape == (T,) and abs(alpha_bar[249] - 0.524) < 0.002 and abs(alpha_bar[499] - 0.0786) < 0.001
assert alpha_bar[-1] < 1e-4
g = torch.Generator().manual_seed(0)
x0 = X[:512]
for t_ in (0, 250, 999):
    t = torch.full((512,), t_)
    xt = q_sample(x0, t, torch.randn(x0.shape, generator=g))
    assert xt.shape == x0.shape
    print(f"t = {t_ + 1:4d}: mean {xt.mean():+.3f} (expected {alpha_bar[t_].sqrt() * x0.mean():+.3f}), std {xt.std():.3f}")
    assert abs(xt.mean() - alpha_bar[t_].sqrt() * x0.mean()) < 0.03
assert 0.95 < xt.std() < 1.05, "at the last step x_t should be unit-variance noise"
tt = torch.tensor([0, 999]); xt2 = q_sample(X[:2], tt, torch.randn(2, 1, S, S, generator=g))
assert (xt2[0] - X[0]).abs().max() < 0.2 and abs(xt2[1].std() - 1) < 0.2, "every image needs its own t: image 0 (t = 1) is almost clean, image 1 (t = 1000) is pure noise"
show(torch.stack([q_sample(X[:1], torch.tensor([t_]), torch.randn(1, 1, S, S, generator=g))[0] for t_ in range(0, 1000, 63)]), "one training image, noised at t = 1, 64, 127, …")
print("✓ exercise 1")

## 2 · A denoiser with a time embedding
The network must know **how noisy** its input is, so the step $t$ goes in as a sinusoidal embedding, exactly like transformer positions (M1.3): for dimension $d$ and $i = 0 \dots d/2 - 1$, frequencies $f_i = 10000^{-i/(d/2)}$, and the embedding is $[\sin(t f_0), \dots, \sin(t f_{d/2-1}), \cos(t f_0), \dots, \cos(t f_{d/2-1})]$.

Write `timestep_embedding(t, dim)` returning shape `(B, dim)`. The denoiser below is given: a residual MLP over the 256 pixels, with the time embedding (through a small MLP) and a learned **class embedding** added to its hidden state. Class index `NULL = 4` means "no label"; it's what classifier-free guidance uses for the unconditional prediction. On a CPU an MLP is several times faster per step than even a small U-Net here; the Experiment section has a U-Net to try on a GPU.

In [ ]:
def timestep_embedding(t, dim=64):
    # TODO: sinusoidal embedding, shape (B, dim): sines first, then cosines
    raise NotImplementedError("Exercise 2")

In [ ]:
class Denoiser(nn.Module):
    # predicts the noise eps from (x_t, t, class label); output has the image's shape
    def __init__(self, hidden=512, depth=2, tdim=64):
        super().__init__()
        self.tdim = tdim
        self.time = nn.Sequential(nn.Linear(tdim, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.label = nn.Embedding(N_CLASSES + 1, hidden)          # +1 for NULL
        self.inp = nn.Linear(S * S, hidden)
        self.blocks = nn.ModuleList([nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.SiLU(),
                                                   nn.Linear(hidden, hidden)) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, S * S))

    def forward(self, x, t, c):
        h = self.inp(x.flatten(1)) + self.time(timestep_embedding(t, self.tdim)) + self.label(c)
        for b in self.blocks:
            h = h + b(h)
        return self.out(h).view(-1, 1, S, S)

**Check 2**

In [ ]:
e = timestep_embedding(torch.tensor([0, 1, 500, 999]), 64)
assert e.shape == (4, 64)
assert torch.allclose(e[0, :32], torch.zeros(32)) and torch.allclose(e[0, 32:], torch.ones(32)), "t = 0: sin 0 = 0, cos 0 = 1"
assert abs(e[1, 0] - math.sin(1.0)) < 1e-6 and abs(e[1, 32] - math.cos(1.0)) < 1e-6
f1 = 10000 ** (-1 / 32)      # the second frequency for dim = 64: 10000^(-i / (dim/2)) with i = 1
assert abs(e[1, 1] - math.sin(f1)) < 1e-6 and abs(e[1, 33] - math.cos(f1)) < 1e-6, "frequencies are 10000^(-i / (dim/2))"
assert (e[2] - e[3]).norm() > 1, "different steps must get different embeddings"
model = Denoiser(hidden=1024 if LONG else 512).to(device)
out = model(X[:8].to(device), torch.randint(0, T, (8,), device=device), y[:8].to(device))
assert out.shape == (8, 1, S, S)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("✓ exercise 2")

## 3 · The training loss, with label dropout
One training step (G1.3 plus G1.4's condition dropout): for a batch of clean images `x0` with labels `c`,
1. draw a step `t` uniformly from `0 … T-1` and noise $\varepsilon \sim \mathcal{N}(0, I)$ for every image,
2. make $x_t$ with `q_sample`,
3. replace each label by `NULL` with probability `p_uncond` (so the same network also learns the unconditional prediction),
4. return the mean squared error between $\varepsilon$ and `model(x_t, t, labels)`.

In [ ]:
def diffusion_loss(model, x0, c, p_uncond=0.15, gen=None):
    # TODO: use gen (a torch.Generator, or None) for all the randomness, e.g. torch.randint(..., generator=gen)
    raise NotImplementedError("Exercise 3")

**Check 3, then train.** A network that outputs zeros scores a loss of exactly 1 (the noise has variance 1), so an untrained model starts near 1. Then the provided loop trains with AdamW and a one-cycle learning-rate schedule. **Target: validation loss below 0.09** after the default 3,000 steps (about a minute on a CPU). The loss never reaches 0: at middle noise levels the network can't tell exactly how much of each pixel is image and how much is noise, so the loss has a floor well above zero. (At very large $t$, where $x_t \approx \varepsilon$, and at tiny $t$, the noise is easy to predict.)

In [ ]:
gen = torch.Generator().manual_seed(0)
with torch.no_grad():
    l0 = diffusion_loss(model, X[:512].to(device), y[:512].to(device), gen=gen).item()
print(f"loss before training: {l0:.3f}")
assert 0.5 < l0 < 2.0

# spy on what the network receives: label dropout and one t per image
seen_t, seen_c = [], []
def spy(xt, t, c):
    seen_t.append(t.clone()); seen_c.append(c.clone()); return torch.zeros_like(xt)
diffusion_loss(spy, X[:256], y[:256], p_uncond=1.0)
diffusion_loss(spy, X[:256], y[:256], p_uncond=0.0)
diffusion_loss(spy, X[:2000], y[:2000], p_uncond=0.15)
assert len(seen_t[0].unique()) > 100, "each image needs its own random t"
assert (seen_c[0] == NULL).all(), "p_uncond = 1 must replace every label by NULL"
assert (seen_c[1] == y[:256]).all(), "p_uncond = 0 must keep every label"
assert 0.10 < (seen_c[2] == NULL).float().mean() < 0.20, "p_uncond = 0.15 should drop about 15% of the labels"

@torch.no_grad()
def val_loss(model, n=1000):
    g = torch.Generator().manual_seed(123)
    return diffusion_loss(model, X_val[:n].to(device), y_val[:n].to(device), p_uncond=0.0, gen=g).item()

STEPS = 20_000 if LONG else 3_000
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=STEPS, pct_start=0.05)
gen = torch.Generator().manual_seed(1)
history, t0 = [], time.time()
for step in range(STEPS):
    idx = torch.randint(0, len(X), (128,), generator=gen)
    loss = diffusion_loss(model, X[idx].to(device), y[idx].to(device), gen=gen)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    history.append(loss.item())
    if step % 500 == 0 or step == STEPS - 1:
        print(f"step {step:5d}  train loss (last 100) {np.mean(history[-100:]):.4f}  val loss {val_loss(model):.4f}  {time.time() - t0:5.1f}s")
model.eval()
vl = val_loss(model)
plt.figure(figsize=(5, 2.5)); plt.plot(np.convolve(history, np.ones(50) / 50, mode="valid")); plt.yscale("log")
plt.xlabel("step"); plt.ylabel("loss (log)"); plt.show()
assert vl < 0.09, f"validation loss {vl:.3f}: train longer or check exercise 3"
print("✓ exercise 3")

## 4 · The DDPM sampler, with classifier-free guidance
Start from $x_T \sim \mathcal{N}(0, I)$ and step $t = T-1, \dots, 0$:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\tilde\varepsilon\right) + \sqrt{\beta_t}\, z, \qquad z \sim \mathcal{N}(0, I)\ \text{(and } z = 0 \text{ on the last step)}$$

with the **guided** noise $\tilde\varepsilon = \varepsilon_u + w(\varepsilon_c - \varepsilon_u)$ from G1.4. `guided_eps` (given) runs the model once on a doubled batch to get both predictions. Write `ddpm_sample`.

To score samples without looking, a tiny CNN **judge** (given) is trained on clean shapes; it is 100% accurate on real validation images. "Class accuracy" = the share of samples the judge files under the class you asked for.

In [ ]:
@torch.no_grad()
def guided_eps(model, x, t, c, w):
    B = x.shape[0]
    tt = torch.full((2 * B,), t, device=x.device, dtype=torch.long)
    e = model(torch.cat([x, x]), tt, torch.cat([c, torch.full_like(c, NULL)]))
    eps_c, eps_u = e[:B], e[B:]
    return eps_u + w * (eps_c - eps_u)

torch.manual_seed(0)
judge = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
                      nn.MaxPool2d(2), nn.Flatten(), nn.Linear(32 * 16, N_CLASSES))
jopt, jg = torch.optim.Adam(judge.parameters(), 1e-3), torch.Generator().manual_seed(0)
for _ in range(400):
    idx = torch.randint(0, len(X), (128,), generator=jg)
    jl = F.cross_entropy(judge(X[idx]), y[idx]); jopt.zero_grad(); jl.backward(); jopt.step()
judge.eval()

@torch.no_grad()
def class_accuracy(samples, c):
    return (judge(samples.detach().cpu().clamp(-1, 1)).argmax(1) == c.cpu()).float().mean().item()

print(f"judge accuracy on real validation shapes: {class_accuracy(X_val, y_val):.1%}")

In [ ]:
@torch.no_grad()
def ddpm_sample(model, c, w=1.0, seed=0):
    # c: labels of shape (B,) on device. Returns x_0 of shape (B, 1, S, S).
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(len(c), 1, S, S, generator=g).to(device)
    # TODO: loop t = T-1 ... 0, using guided_eps and the update above;
    #       draw z with torch.randn(..., generator=g) and skip it on the last step
    raise NotImplementedError("Exercise 4")
    return x

**Check 4.** 128 samples, 32 per class, 1,000 steps each. Targets: with $w = 0$ (unconditional) about chance, 25%; with $w = 3$ **at least 85%** of samples land in the requested class. Look at the pictures too: blurry edges are normal for a model this small, but you should be able to name every shape.

In [ ]:
c = (torch.arange(128) % N_CLASSES).to(device)
t0 = time.time(); x_w0 = ddpm_sample(model, c, w=0.0); x_w3 = ddpm_sample(model, c, w=3.0)
print(f"two runs of 1,000 steps: {time.time() - t0:.1f}s")
acc0, acc3 = class_accuracy(x_w0, c), class_accuracy(x_w3, c)
print(f"class accuracy: w = 0 → {acc0:.0%},  w = 3 → {acc3:.0%}")
print("unconditional class mix:", torch.bincount(judge(x_w0.cpu().clamp(-1, 1)).argmax(1), minlength=4).tolist())
show(x_w0, "DDPM, w = 0 (unconditional)"); show(x_w3, "DDPM, w = 3 (disk, square, triangle, plus, …)")
assert x_w3.shape == (128, 1, S, S) and torch.isfinite(x_w3).all()
assert acc0 < 0.5, "w = 0 ignores the label, so accuracy should be near chance"
assert acc3 >= 0.85, "guided samples should mostly match their label"
print("✓ exercise 4")

## 5 · DDIM: the same model, 50× fewer steps
DDIM (G1.3) jumps from step $t$ straight to an earlier step $t'$ with no fresh noise:

$$\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\tilde\varepsilon}{\sqrt{\bar\alpha_t}}, \qquad x_{t'} = \sqrt{\bar\alpha_{t'}}\;\hat x_0 + \sqrt{1-\bar\alpha_{t'}}\;\tilde\varepsilon$$

Use `steps` evenly spaced time steps from $T-1$ down to 0 (`torch.linspace(T - 1, 0, steps).round().long()`). One practical fix, used in the original DDPM code too: **clip** $\hat x_0$ to $[-1, 1]$ (the data range), then recompute $\tilde\varepsilon = (x_t - \sqrt{\bar\alpha_t}\,\hat x_0)/\sqrt{1-\bar\alpha_t}$ so the two stay consistent. Without it, early steps (where $\sqrt{\bar\alpha_t} \approx 0.006$) blow small errors up into garbage. On the final step, return $\hat x_0$.

In [ ]:
@torch.no_grad()
def ddim_sample(model, c, w=1.0, steps=20, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(len(c), 1, S, S, generator=g).to(device)
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    # TODO: for each t (and the next t'), predict, clip x0_hat, recompute eps, jump to t'
    raise NotImplementedError("Exercise 5")

**Check 5.** Same labels, $w = 3$, 20 steps instead of 1,000. Target: class accuracy **at least 80%**, and the same seed gives the same images (DDIM is deterministic).

In [ ]:
t0 = time.time(); xd = ddim_sample(model, c, w=3.0, steps=20); dt = time.time() - t0
accd = class_accuracy(xd, c)
print(f"DDIM, 20 steps: class accuracy {accd:.0%} in {dt:.2f}s (DDPM: {acc3:.0%} with 1,000 steps)")
show(xd, "DDIM, 20 steps, w = 3")
assert torch.equal(xd, ddim_sample(model, c, w=3.0, steps=20)), "DDIM should be deterministic for a fixed seed"
assert xd.min() >= -1 and xd.max() <= 1
assert accd >= 0.80
print("✓ exercise 5")

## Experiment
- **Guidance sweep.** The cell below measures class accuracy and the share of "clean" pixels (within 0.25 of ±1, as in the training data) for several $w$, using 50-step DDIM. Where does accuracy saturate, and what happens to the images at large $w$?
- **No label dropout.** Retrain with `p_uncond=0.0`. What does $w = 3$ do now? (The "unconditional" prediction was never trained.)
- **Steps.** Try DDIM with 5, 10 and 50 steps. How few can you get away with?
- **Wraparound.** In `ddim_sample`, replace the last step's `return x0` by a jump to `alpha_bar[-1]` (what you'd write for "the step after 0"). Class accuracy drops from about 88% to about chance (28% in the reference run), because in Python `alpha_bar[-1]` is the *last* element, $\bar\alpha_{1000} \approx 4 \times 10^{-5}$, i.e. pure noise, not "clean". DDPM never needs that index.
- **On a GPU:** set `LONG = True`, or swap in the small U-Net below (`model = TinyUNet().to(device)`) and retrain. Convolutions share weights across positions, which is why U-Nets win on real images.

In [ ]:
def clean_share(x):
    return ((x.abs() - 1).abs() < 0.25).float().mean().item()

print(f"real data: clean pixels {clean_share(X_val):.0%}")
for w in [0.0, 1.0, 2.0, 3.0, 5.0, 8.0]:
    xs = ddim_sample(model, c, w=w, steps=50)
    print(f"w = {w:3.0f}: class accuracy {class_accuracy(xs, c):4.0%}   clean pixels {clean_share(xs):4.0%}")

class TinyUNet(nn.Module):
    # optional: a small U-Net (16 → 8 → 4 → 8 → 16) with the same interface; slow on a CPU
    def __init__(self, ch=(32, 64, 128), tdim=64):
        super().__init__(); a, b, m = ch; self.tdim = tdim
        self.emb = nn.Sequential(nn.Linear(tdim, 128), nn.SiLU(), nn.Linear(128, 128)); self.label = nn.Embedding(N_CLASSES + 1, 128)
        self.proj = nn.ModuleList([nn.Linear(128, k) for k in (a, b, m, b, a)])
        self.inp = nn.Conv2d(1, a, 3, padding=1)
        self.c = nn.ModuleList([nn.Conv2d(k, k, 3, padding=1) for k in (a, b, m, b, a)])
        self.down1, self.down2 = nn.Conv2d(a, b, 3, 2, 1), nn.Conv2d(b, m, 3, 2, 1)
        self.up2, self.up1 = nn.ConvTranspose2d(m, b, 2, 2), nn.ConvTranspose2d(b, a, 2, 2)
        self.out = nn.Conv2d(a, 1, 3, padding=1)
    def block(self, i, h, e):
        return h + self.c[i](F.silu(h + self.proj[i](e)[:, :, None, None]))
    def forward(self, x, t, c):
        e = self.emb(timestep_embedding(t, self.tdim)) + self.label(c)
        h1 = self.block(0, self.inp(x), e); h2 = self.block(1, self.down1(h1), e); h3 = self.block(2, self.down2(h2), e)
        u = self.block(3, self.up2(h3) + h2, e); u = self.block(4, self.up1(u) + h1, e)
        return self.out(F.silu(u))

print("TinyUNet output shape:", tuple(TinyUNet()(X[:2], torch.tensor([0, 1]), y[:2]).shape))

## Recap
- Training is regression: noise an image in one jump, predict the noise, squared error; label dropout gives you the unconditional model for free.
- DDPM sampling walks back through all 1,000 steps; classifier-free guidance pushes each step toward the label at the cost of a second forward pass (batched here).
- DDIM reuses the same network with deterministic jumps: 20 steps instead of 1,000. Clipping $\hat x_0$ to the data range keeps big jumps stable.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, eps):
    ab = alpha_bar.to(x0.device)[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def timestep_embedding(t, dim=64):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=1)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def diffusion_loss(model, x0, c, p_uncond=0.15, gen=None):
    B = x0.shape[0]
    t = torch.randint(0, T, (B,), generator=gen).to(x0.device)
    eps = torch.randn(x0.shape, generator=gen).to(x0.device)
    xt = q_sample(x0, t, eps)
    drop = (torch.rand(B, generator=gen) < p_uncond).to(x0.device)
    c = torch.where(drop, torch.full_like(c, NULL), c)
    return F.mse_loss(model(xt, t, c), eps)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
@torch.no_grad()
def ddpm_sample(model, c, w=1.0, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(len(c), 1, S, S, generator=g).to(device)
    for t in range(T - 1, -1, -1):
        eps = guided_eps(model, x, t, c, w)
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn(x.shape, generator=g).to(device)
    return x

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
@torch.no_grad()
def ddim_sample(model, c, w=1.0, steps=20, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(len(c), 1, S, S, generator=g).to(device)
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    for i, t in enumerate(ts):
        ab = alpha_bar[t]
        eps = guided_eps(model, x, t, c, w)
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)
        if i == len(ts) - 1:
            return x0
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()
        ab_next = alpha_bar[ts[i + 1]]
        x = ab_next.sqrt() * x0 + (1 - ab_next).sqrt() * eps